First we look at the process model 


In [2]:
import control as ct
import matplotlib.pyplot as plt
import numpy as np

S = .10 # 1e7 mu s / 1e9 ppb = 1e-2 

z = ct.tf('z')
G = S/(1-z**(-1))
plt.figure(1)
ct.pzmap(G)
plt.title("Pole/zero plot for plant")
# plt.figure(2)
# ct.bode_plot(G, display_margins=True)
# plt.title("Bode plot of plant")
# plt.show()


Text(0.5, 1.0, 'Pole/zero plot for plant')


In [43]:
# hr = np.linspace(0, 50000, 10000)
# Gi = 1/( 1 - z**(-1))
# ct.rlocus(G*Gi , gains=hr)
# plt.title("Possible conjugate pairs of poles using I-controller")
# plt.show()


Now find the Kp and Ki using pole placement


In [3]:
import sympy as sp
import numpy as np

S = sp.symbols('S')
def auto_place_pi(desired_poles):
    
    z, Kp, Ki = sp.symbols('z Kp Ki')
    # Define the Transfer Function components
    plant_num = S * z
    plant_den = z - 1
    pi_num = Kp * (z - 1) + Ki * z
    pi_den = z - 1
    
    # Build the Characteristic Polynomial: (Plant_Den * PI_Den) + (Plant_Num * PI_Num) = 0
    char_eq = sp.Poly(plant_den * pi_den + plant_num * pi_num, z)
    actual_coeffs = char_eq.all_coeffs()
    # print("actual coeffs:")
    # sp.pprint(actual_coeffs)
    
    # Get the target polynomial coefficients from the desired poles
    desired_poly = np.poly(desired_poles)
    
    # Set up equations matching Actual coefficients to Desired coefficients
    # (We divide by actual_coeffs[0] to normalize the polynomial so the highest power is 1)
    equations = []
    for actual, desired in zip(actual_coeffs, desired_poly):
        equations.append(sp.Eq(actual / actual_coeffs[0], desired))
        
    # Let SymPy solve the algebra
    solution = sp.solve(equations, (Kp, Ki), dict=True)[0]
    return (solution[Kp]), (solution[Ki])




In [4]:
desired_poles = [0.3 + 0.1j, 0.3 - 0.1j]

Kp_repr, Ki_repr = auto_place_pi(desired_poles)

print(f"SymPy Calculated Kp: {Kp_repr}")
print(f"SymPy Calculated Ki: {Ki_repr}")


SymPy Calculated Kp: 4.0/S
SymPy Calculated Ki: 5.0/S


In [5]:
S_nom = 0.1

Kp_calc = float(Kp_repr.subs(S, S_nom))
Ki_calc = float(Ki_repr.subs(S, S_nom))

print(f"Calculated Kp: {Kp_calc:.4f}")
print(f"Calculated Ki: {Ki_calc:.4f}")


Calculated Kp: 40.0000
Calculated Ki: 50.0000


Now place the poles


In [19]:
Kp = 40
Ki = 50

def pzmap_with_ks(Kp, Ki, label=None):
    Gpi = (Kp*(z-1) + Ki*z)/(z-1)
    Gcl = ct.feedback(G*Gpi, 1)
    
    print(f"\n--- Kp={Kp}, Ki={Ki} ---") 
    print("Poles:", ct.poles(Gcl))
    
    plt.figure(1)
    ct.pzmap(Gcl)
    
    legend_label = label if label else rf"$K_p={Kp}, K_i={Ki}$"
    plt.plot([], [], marker='x', linestyle='none', label=legend_label)
    
pzmap_with_ks(Kp, Ki)
# plt.figure(2)
# ct.bode_plot(G*Gpi, display_margins=True)
# plt.title("Bode plot of closed loop")
# plt.show()


--- Kp=40, Ki=50 ---
Poles: [0.3+0.1j 0.3-0.1j]


In [7]:
def get_ks(desired_poles) -> tuple[float, float]:
    Kp_repr, Ki_repr = auto_place_pi(desired_poles)
    S_nom = 0.1
    
    Kp_calc = float(Kp_repr.subs(S, S_nom))
    Ki_calc = float(Ki_repr.subs(S, S_nom))
    return (Kp_calc, Ki_calc)


In [8]:
desired_poles = [0.3 + 0.3j, 0.3 - 0.3j]
(kp, ki) = get_ks(desired_poles)
print(f"{desired_poles=}\n\t{kp = :.2f}, {ki = :.2f}")

pzmap_with_ks(kp, ki)


desired_poles=[(0.3+0.3j), (0.3-0.3j)]
	kp = 13.33, ki = 32.22
<TransferFunction>: sys[43]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = True
    4.556 z^2 - 1.333 z
  -----------------------
  5.556 z^2 - 3.333 z + 1
[0.3+0.3j 0.3-0.3j]


In [9]:
desired_poles = [0.1 + 0.1j, 0.1 - 0.1j]

(kp, ki) = get_ks(desired_poles)

print(f"{desired_poles=}\n\t{kp = :.2f}, {ki = :.2f}")
pzmap_with_ks(kp, ki)


desired_poles=[(0.1+0.1j), (0.1-0.1j)]
	kp = 80.00, ki = 410.00
<TransferFunction>: sys[56]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = True
    49 z^2 - 8 z
  -----------------
  50 z^2 - 10 z + 1
[0.1+0.1j 0.1-0.1j]


In [11]:
kp, ki = 1, 1
pzmap_with_ks(kp, ki)


<TransferFunction>: sys[82]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = True
    0.2 z^2 - 0.1 z
  -------------------
  1.2 z^2 - 2.1 z + 1
[0.875+0.26020825j 0.875-0.26020825j]


In [26]:
plt.figure(1)
plt.clf() 

pzmap_with_ks(1.0, 1.0, label=r"$K_p=1,K_i=1$")
pzmap_with_ks(40, 50, label=r"$K_p=40,K_i=50$")
pzmap_with_ks(10, 50, label=r"$K_p=10,K_i=50$")
pzmap_with_ks(13, 32)
pzmap_with_ks(13, 4.59)
pzmap_with_ks(100, 500)

# Get all handles and labels from the plot
handles, labels = plt.gca().get_legend_handles_labels()

# Filter out the automatic 'sys[...]' labels generated by ct.pzmap
clean_handles = []
clean_labels = []
for h, l in zip(handles, labels):
    if not l.startswith("sys["):
        clean_handles.append(h)
        clean_labels.append(l)

# Display the legend using only our clean labels, and move it to the upper left
plt.legend(clean_handles, clean_labels, loc='upper left')
plt.title("Pole/zero plot of PI-controllers")
plt.suptitle("")

plt.savefig("pzplot_pi_controllers.png", dpi=300, bbox_inches='tight')
plt.show()


--- Kp=1.0, Ki=1.0 ---
Poles: [0.875+0.26020825j 0.875-0.26020825j]
--- Kp=40, Ki=50 ---
Poles: [0.3+0.1j 0.3-0.1j]
--- Kp=10, Ki=50 ---
Poles: [0.21428571+0.31134992j 0.21428571-0.31134992j]
--- Kp=13, Ki=32 ---
Poles: [0.3+0.30301515j 0.3-0.30301515j]
--- Kp=13, Ki=4.59 ---
Poles: [0.59804277+0.06924601j 0.59804277-0.06924601j]
--- Kp=100, Ki=500 ---
Poles: [0.09836066+0.08196721j 0.09836066-0.08196721j]
